In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00 - Setup e Discovery das Fontes
# MAGIC
# MAGIC Objetivo:
# MAGIC - Organizar os arquivos fonte
# MAGIC - Validar o caminho do Volume
# MAGIC - Explorar schemas e amostras
# MAGIC - Identificar problemas de qualidade dos dados

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [0]:
BASE_PATH = "/Volumes/case/default/case_data"

SOURCE_PATH = f"{BASE_PATH}/sources"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print("BASE_PATH:", BASE_PATH)
print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)

BASE_PATH: /Volumes/case/default/case_data
SOURCE_PATH: /Volumes/case/default/case_data/sources
BRONZE_PATH: /Volumes/case/default/case_data/bronze
SILVER_PATH: /Volumes/case/default/case_data/silver
GOLD_PATH: /Volumes/case/default/case_data/gold


In [0]:
# Criando diretórios do projeto
for path in [SOURCE_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    dbutils.fs.mkdirs(path)

display(dbutils.fs.ls(BASE_PATH))

path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/bronze/,bronze/,0,1781140251768
dbfs:/Volumes/case/default/case_data/gold/,gold/,0,1781140251768
dbfs:/Volumes/case/default/case_data/silver/,silver/,0,1781140251768
dbfs:/Volumes/case/default/case_data/sources/,sources/,0,1781140251768


In [0]:
# Esta célula pode ser executada mais de uma vez sem quebrar o notebook.
# Ela move os arquivos para /sources apenas se ainda estiverem na raiz do volume.

arquivos_fonte = [
    "atendimento_ocorrencias.ndjson",
    "cadastro_produtos_api_dump.json",
    "comercial_canais.xlsx",
    "crm_clientes_export.xlsx",
    "erp_pedidos_cabecalho_2025.csv",
    "erp_pedidos_itens_2025.csv",
    "legado_regioes_pipe.txt",
    "logistica_entregas.json",
    "vendedores.csv"
]

for arquivo in arquivos_fonte:
    origem = f"{BASE_PATH}/{arquivo}"
    destino = f"{SOURCE_PATH}/{arquivo}"
    
    try:
        # Se o arquivo já existe em sources, apenas informa.
        dbutils.fs.ls(destino)
        print(f"Já está em sources: {arquivo}")
    except:
        try:
            dbutils.fs.mv(origem, destino)
            print(f"Movido para sources: {arquivo}")
        except Exception as e:
            print(f"Arquivo não encontrado na origem ou já tratado: {arquivo}")

Já está em sources: atendimento_ocorrencias.ndjson
Já está em sources: cadastro_produtos_api_dump.json
Já está em sources: comercial_canais.xlsx
Já está em sources: crm_clientes_export.xlsx
Já está em sources: erp_pedidos_cabecalho_2025.csv
Já está em sources: erp_pedidos_itens_2025.csv
Já está em sources: legado_regioes_pipe.txt
Já está em sources: logistica_entregas.json
Já está em sources: vendedores.csv


In [0]:
display(dbutils.fs.ls(SOURCE_PATH))

path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/sources/atendimento_ocorrencias.ndjson,atendimento_ocorrencias.ndjson,39129,1781139762000
dbfs:/Volumes/case/default/case_data/sources/cadastro_produtos_api_dump.json,cadastro_produtos_api_dump.json,28302,1781139763000
dbfs:/Volumes/case/default/case_data/sources/comercial_canais.xlsx,comercial_canais.xlsx,5180,1781139764000
dbfs:/Volumes/case/default/case_data/sources/crm_clientes_export.xlsx,crm_clientes_export.xlsx,16938,1781139765000
dbfs:/Volumes/case/default/case_data/sources/erp_pedidos_cabecalho_2025.csv,erp_pedidos_cabecalho_2025.csv,56007,1781139766000
dbfs:/Volumes/case/default/case_data/sources/erp_pedidos_itens_2025.csv,erp_pedidos_itens_2025.csv,40014,1781139767000
dbfs:/Volumes/case/default/case_data/sources/legado_regioes_pipe.txt,legado_regioes_pipe.txt,305,1781139768000
dbfs:/Volumes/case/default/case_data/sources/logistica_entregas.json,logistica_entregas.json,123545,1781139769000
dbfs:/Volumes/case/default/case_data/sources/vendedores.csv,vendedores.csv,1752,1781139770000


In [0]:
pedidos_cabecalho_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/erp_pedidos_cabecalho_2025.csv")
)

pedidos_itens_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/erp_pedidos_itens_2025.csv")
)

vendedores_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/vendedores.csv")
)

regioes_raw = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .option("inferSchema", True)
    .csv(f"{SOURCE_PATH}/legado_regioes_pipe.txt")
)

produtos_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{SOURCE_PATH}/cadastro_produtos_api_dump.json")
)

entregas_raw = (
    spark.read
    .option("multiLine", True)
    .json(f"{SOURCE_PATH}/logistica_entregas.json")
)

ocorrencias_raw = (
    spark.read
    .json(f"{SOURCE_PATH}/atendimento_ocorrencias.ndjson")
)

In [0]:
clientes_pdf = pd.read_excel(f"{SOURCE_PATH}/crm_clientes_export.xlsx")
canais_pdf = pd.read_excel(f"{SOURCE_PATH}/comercial_canais.xlsx")

clientes_raw = spark.createDataFrame(clientes_pdf.astype(str))
canais_raw = spark.createDataFrame(canais_pdf.astype(str))

In [0]:
fontes = {
    "pedidos_cabecalho_raw": pedidos_cabecalho_raw,
    "pedidos_itens_raw": pedidos_itens_raw,
    "vendedores_raw": vendedores_raw,
    "regioes_raw": regioes_raw,
    "produtos_raw": produtos_raw,
    "entregas_raw": entregas_raw,
    "ocorrencias_raw": ocorrencias_raw,
    "clientes_raw": clientes_raw,
    "canais_raw": canais_raw
}

for nome, df in fontes.items():
    print("=" * 100)
    print(nome)
    print("Quantidade de linhas:", df.count())
    df.printSchema()
    display(df.limit(5))

pedidos_cabecalho_raw
Quantidade de linhas: 403
root
 |-- order_id: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- promised_date: string (nullable = true)
 |-- status_order: string (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- payment_details: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



order_id,customer_code,seller_id,order_date,promised_date,status_order,gross_amount,discount_amount,net_amount,payment_details,last_update
O00001,C0002,V036,2025-07-31,06/08/2025,Faturado,10788.64,0.0,10788.64,"""{""""priority"""": null, """"source"""": """"APP""""}""",2025-08-20T00:00:00.000Z
O00002,C0023,V008,2025/04/13,2025-04-26,EM_SEPARACAO,14462.32,0.0,14462.32,"""{""""priority"""": """"low"""", """"source"""": """"ERP""""}""",2025-04-26T00:00:00.000Z
O00003,C0127,V012,2025/04/01,16/04/2025,faturado,13802.56,1994.35,11808.21,"""{""""priority"""": """"low"""", """"source"""": """"APP""""}""",2025-04-19T00:00:00.000Z
O00004,C0165,V035,2025/11/05,07/11/2025,cancelado,13933.98,670.75,13263.23,"""{""""priority"""": null, """"source"""": """"APP""""}""",2025-11-11T00:00:00.000Z
O00005,C0048,V001,2026/02/19,21/02/2026,null,5027.19,0.0,5027.19,"""{""""priority"""": """"high"""", """"source"""": """"Portal""""}""",2026-02-25T00:00:00.000Z


pedidos_itens_raw
Quantidade de linhas: 995
root
 |-- order_id: string (nullable = true)
 |-- item_seq: integer (nullable = true)
 |-- product_code: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- total_item: double (nullable = true)
 |-- item_status: string (nullable = true)



order_id,item_seq,product_code,quantity,unit_price,total_item,item_status
O00001,1,P0027,5.0,453.12,2265.6,Ativo
O00001,2,P0035,10.0,2278.1,22781.0,null
O00001,3,P0063,2.0,629.8,1259.6,Ativo
O00001,4,P0001,3.0,1502.54,4507.62,Ativo
O00002,1,P0032,3.0,"1274,78",3824.34,Ativo


vendedores_raw
Quantidade de linhas: 42
root
 |-- seller_id: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- canal_id: string (nullable = true)
 |-- regional_code: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- status: string (nullable = true)



...

created_at,customer_code,event_type,metadata,order_id,severity,status,ticket_id
2025-01-04 05:00:00,null,refund,null,O00385,High,closed,T00001
2026/02/12,null,troca,null,O00051,medium,open,T00002
2026/02/07,null,refund,null,O00181,null,null,T00003
2025/12/18,null,delay,null,O00371,low,null,T00004
27/02/2025 16:00,null,null,null,O00055,medium,Open,T00005


clientes_raw
Quantidade de linhas: 183
root
 |-- customer_id: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- porte: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: string (nullable = true)
 |-- email: string (nullable = true)
 |-- status_cliente: string (nullable = true)
 |-- updated_at: string (nullable = true)



customer_id,nome_cliente,segmento,porte,cidade,estado,data_cadastro,email,status_cliente,updated_at
C0001,Cliente 1,nan,Grande,Florianópolis,santa catarina,2024-01-26,cliente1@empresa.com,nan,2025-02-22 00:00:00
C0002,Cliente 2,Educação,grande,Curitiba,Paraná,2024-03-30,cliente2@empresa.com,Ativo,2025-04-30 00:00:00
C0003,Cliente 3,Varejo,nan,Curitiba,PR,2025/09/08,cliente3@empresa.com,nan,2025-10-07 00:00:00
C0004,Cliente 4,Saúde,nan,Uberlândia,Minas Gerais,2024-08-13,cliente4@empresa.com,nan,2025-08-05 00:00:00
C0005,Cliente 5,Indústria,Média,Niterói,Rio de Janeiro,2024-10-11,cliente5@empresa.com,Inativo,2025-02-19 00:00:00


canais_raw
Quantidade de linhas: 8
root
 |-- id_canal: string (nullable = true)
 |-- nome_canal: string (nullable = true)
 |-- tipo_canal: string (nullable = true)
 |-- ativo: string (nullable = true)
 |-- observacao: string (nullable = true)



id_canal,nome_canal,tipo_canal,ativo,observacao
CH01,Inside Sales,Direto,sim,nan
CH02,Parceiros,Indireto,Sim,nan
CH03,Marketplace,Digital,SIM,nan
CH04,Field Sales,Direto,nao,canal legado
CH05,E-commerce,digital,sim,nan
